# Урок 11 - Протокол контексту моделі (MCP)

**Протокол контексту моделі (MCP)** — це відкритий стандарт, який дозволяє агентам динамічно знаходити та використовувати інструменти, ресурси та джерела даних під час роботи. Замість жорсткого кодування інструментів у агенті, MCP дає змогу агентам підключатися до зовнішніх серверів, які надають функціональні можливості на вимогу.

У цьому уроці ви дізнаєтесь:
- Що таке MCP і чому він важливий для систем агентів
- Як працює клієнт-серверна архітектура MCP
- Як створювати агентів, які використовують виявлення інструментів у стилі MCP


## Налаштування

**Передумови:**
- Проєкт Microsoft Foundry з розгорнутою моделлю
- Виконайте `az login` для автентифікації


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from typing import Annotated
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Що таке Протокол Контексту Моделі (MCP)?

MCP визначає стандартний спосіб для AI агентів знаходити та взаємодіяти із зовнішніми інструментами та джерелами даних:

- **MCP Сервер**: Надає інструменти, ресурси та підказки через стандартний протокол
- **MCP Клієнт**: Середовище виконання агента, яке підключається до серверів і виявляє доступні можливості
- **Динамічне Виявлення**: Агенти не потребують жорстко закодованих інструментів — вони знаходять доступні на час виконання

Це потужно для побудови розширюваних систем агентів, де нові можливості можна додавати без зміни коду агента.


## Як працює MCP

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. Аґент (клієнт MCP) підключається до сервера MCP
2. Сервер відповідає списком доступних інструментів та їх схем
3. Аґент може викликати будь-який знайдений інструмент під час свого розмірковування
4. Результати передаються назад через той самий протокол


## Імітація виявлення інструментів MCP

Оскільки справжній сервер MCP вимагає робочого серверного процесу, ми продемонструємо шаблон, використовуючи функції `@tool`, які імітують те, що надав би підключений до MCP сервіс розміщення.

У продакшені ці інструменти динамічно виявлятимуться з сервера MCP, а не визначатимуться локально.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## Побудова агента з інструментами стилю MCP


In [ ]:
agent = client.as_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP у виробництві

У виробничому середовищі MCP забезпечує потужні патерни:

- **Динамічне виявлення інструментів**: Агенти підключаються до серверів MCP і виявляють інструменти під час виконання
- **Розділена архітектура**: Постачальники інструментів можуть оновлюватися незалежно від агента
- **Обмін між організаціями**: Команди можуть надавати можливості через сервери MCP, які може використовувати будь-який агент
- **Підтримка Microsoft Agent Framework**: MAF має вбудовану підтримку клієнта MCP через інтеграцію `mcp`

Для використання реального сервера MCP з MAF слід підключатися через `hosted_mcp_tool()` або інтеграцію клієнта MCP.

**Дізнатись більше:**
- [Специфікація MCP](https://modelcontextprotocol.io/)
- [Підтримка MCP у Microsoft Agent Framework](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## Резюме

У цьому уроці ви дізналися:
- **MCP** — це відкритий стандарт для динамічного виявлення інструментів між агентами та постачальниками інструментів
- **Клієнт-серверна архітектура** дозволяє агентам відкривати можливості під час виконання
- MCP забезпечує **розширювані, розділені системи агентів**, де інструменти можна додавати без змін коду
- Microsoft Agent Framework надає **вбудовану підтримку MCP** для продуктивного використання


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Відмова від відповідальності**:
Цей документ було перекладено за допомогою сервісу штучного інтелекту для перекладу [Co-op Translator](https://github.com/Azure/co-op-translator). Хоча ми прагнемо до точності, будь ласка, майте на увазі, що автоматичні переклади можуть містити помилки або неточності. Оригінальний документ рідною мовою слід вважати авторитетним джерелом. Для критично важливої інформації рекомендується професійний людський переклад. Ми не несемо відповідальності за будь-які непорозуміння або неправильні тлумачення, що виникли внаслідок використання цього перекладу.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
